# M0.5 — Veyra3 Knowledge-Displacement Experiment (Colab Runner)

Runs the full M0.5 pipeline (Phase A → B → C) end-to-end on a Colab GPU.

**Total expected runtime:** ~1.5–3 h on a T4 (Free) or ~15–45 min on A100.

**What this notebook does:**
1. Clones `localsparse` repo + installs deps
2. Runs the pre-benchmark (baseline Veyra3 stats)
3. **Phase A** — surgery + G1 (no NaN) + G2 (branch non-collapse) + G3 (indexer routing) + Gsave (save/load)
4. **Capacity sweep** — finds the knee that calibrates G4/G6 thresholds
5. **Phase B** — G4 (single-wks recall), G5 (multi-wks routing), G9 (agent smoke)
6. **Phase C** — **G6 knowledge displacement** (the headline experiment)
7. Renders all results inline

**Decision rule (plan §6.12):** If G6 ≥ 0.6 → proceed to MiniCPM5 M1 with high confidence.

In [ ]:
# 1. Setup
import os, subprocess, sys, json
from pathlib import Path

REPO_URL = os.environ.get('LOCALSPARSE_REPO', 'https://github.com/kaaninel/localsparse')
REPO_DIR = Path('/content/localsparse')
if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -e . transformers==5.9.0 accelerate
sys.path.insert(0, str(REPO_DIR))
import torch
print('cuda?', torch.cuda.is_available(), 'device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
OUT = Path('/content/runs/m05')
OUT.mkdir(parents=True, exist_ok=True)
MODEL = 'veyra-ai/veyra3-5m-base'
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'OUT={OUT} MODEL={MODEL} DEV={DEV}')

## 2. Pre-benchmark (baseline) — ~2 min

In [ ]:
!python scripts/veyra_pre_benchmark.py --model {MODEL} --out {OUT} --device {DEV} --ctx-lens 1024,2048,4096
print(json.dumps(json.loads((OUT/'pre_benchmark.json').read_text()), indent=2)[:2000])

## 3. Phase A — surgery + numerical/branch/indexer gates — ~15–30 min

In [ ]:
!python scripts/run_phase_a_gates.py --model {MODEL} --out {OUT} --device {DEV} --steps 1000 --batch 4 --seq-len 1024 --g3-trials 64 --g3-blocks 16

## 4. Capacity sweep — calibrate G4/G6 thresholds — ~30–60 min

In [ ]:
!python scripts/factoid_capacity_sweep.py --model {MODEL} --out {OUT} --device {DEV} --n-list 64,128,256,512,1024 --batch 16 --seq-len 1024 --epochs 60 --lr 1e-3 --repeats 80
print((OUT / 'capacity_sweep' / 'derived_thresholds.json').read_text())

## 5. Phase B — workspace gates (G4, G5, G9) — ~20–40 min

In [ ]:
!python scripts/run_phase_b_gates.py --model {MODEL} --out {OUT} --device {DEV} --n-facts 256 --batch 16 --seq-len 1024 --epochs 60 --train-repeats 80 --n-partitions 8

## 6. Phase C — **knowledge displacement G6 (HEADLINE)** — ~30–60 min

In [ ]:
!python scripts/run_knowledge_displacement.py --model {MODEL} --out {OUT} --device {DEV} --n-facts 200 --batch 16 --seq-len 1024 --epochs 60 --train-repeats 80

## 7. Inline results aggregation

In [ ]:
import pandas as pd
rows = []
for run_dir in OUT.glob('phase_*'):
    gp = run_dir / 'gates.jsonl'
    if gp.exists():
        for line in gp.read_text().splitlines():
            rec = json.loads(line); rec['run'] = run_dir.name
            rows.append(rec)
df = pd.DataFrame(rows)
df[['run', 'gate_id', 'metric', 'value', 'threshold', 'status']] if not df.empty else 'no gate records'

In [ ]:
# Verdict on G6
g6_rows = df[df['gate_id'] == 'G6'] if not df.empty else pd.DataFrame()
if not g6_rows.empty:
    g6 = g6_rows.iloc[-1]
    print(f'=== G6 HEADLINE ===')
    print(f'  mount/weights ratio: {g6["value"]:.3f}')
    print(f'  threshold:           {g6["threshold"]:.3f}')
    print(f'  status:              {g6["status"].upper()}')
    print()
    if g6['status'] == 'pass':
        print('🚀  PROCEED to MiniCPM5 M1 with HIGH CONFIDENCE')
    elif g6['status'] == 'stretch':
        print('⚠️   PROCEED with flagged risk — mount mechanism may need redesign at scale')
    else:
        print('🛑  STOP — redesign mount mechanism before MiniCPM5')